In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q04/q04_rewrite/checkpoints/pre_cell_2.pickle

In [ ]:
%%cudf.pandas.profile
### cell 2 ###

# Extract unique L_ORDERKEY values where commit date is before receipt date
fline_keys = (
    lineitem
    .loc[lineitem.L_COMMITDATE < lineitem.L_RECEIPTDATE, ['L_ORDERKEY']]
    .drop_duplicates()
    .rename(columns={'L_ORDERKEY': 'O_ORDERKEY'})
)

# Filter orders by date range, join to filtered lineitem keys, and count orders per priority
total = (
    orders
    .loc[
        (orders.O_ORDERDATE >= '1993-08-01') &
        (orders.O_ORDERDATE < '1993-11-01')
    ]
    .merge(fline_keys, on='O_ORDERKEY', how='inner')
    .groupby('O_ORDERPRIORITY')
    .size()
    .reset_index(name='O_ORDERKEY')
)